# 03 — Phase 3 · Expérience 1 — Binaire vs multiclasses (H1, Q2)

## Question
Un détecteur binaire qui semble excellent reste-t-il utile lorsque la tâche consiste à identifier la famille d’attaque ?  C’est exactement l’objet de Q2.

## Hypothèse H1
La classification multiclasses est plus difficile que la classification binaire.

## Protocole
- Mêmes lignes dédupliquées, même split stratifié 80/20 par famille, même graine (42), mêmes trois pipelines que la baseline.  Seule la cible change.
- `binary` : Benign vs DDoS.
- `multiclass` : Benign + 12 types d’attaque CIC-DDoS2019, obtenus en fusionnant `DrDoS_X` avec `X` (13 classes).
- `multiclass_no_protocol` : même cible, `Protocol` retiré.  Protocol est presque déterminant pour la famille ; cette ablation mesure donc la part du score multiclasses qui relève d’une lecture du protocole.
- Ce notebook exécute lui-même les 9 fits et écrit `results/exp1.csv` ; il n’y a ni module ni CLI.


In [ ]:
import json
import time
import warnings
from pathlib import Path
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix as sk_confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent
DATA = ROOT / "dataset"
RES = ROOT / "results"
FIG = ROOT / "figures"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

BENIGN = "Benign"
SEEDS = [42, 43, 44]
CAP = 20_000


def load_dataset(data_dir=None):
    files = sorted(Path(data_dir or DATA).glob("*.parquet"))
    frames = []
    for path in files:
        frame = pd.read_parquet(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        frame["src_file"] = path.name
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    raw["Label"] = raw["Label"].astype(str).str.strip().replace({"UDP-lag": "UDPLag"})
    features = [column for column in raw.columns if column not in ("Label", "src_file")]
    X = raw[features].astype("float64")
    y_family = raw["Label"].astype("string").rename("family")
    y_binary = (y_family != BENIGN).astype("int8").rename("binary")
    meta = pd.DataFrame(
        {"src_file": raw["src_file"].astype("string"), "label": y_family.to_numpy()},
        index=X.index,
    )
    return X, y_family, y_binary, meta


def dedupe(X, y, meta=None):
    key = pd.Series(pd.util.hash_pandas_object(X, index=False).to_numpy(), index=X.index)
    grouped = y.groupby(key, sort=False)
    labels = grouped.first()
    n_labels = grouped.nunique()
    conflicts = n_labels[n_labels > 1].index
    if len(conflicts):
        subset = pd.DataFrame({"hash": key, "y": y}).loc[lambda frame: frame["hash"].isin(conflicts)]
        majority = subset.groupby("hash")["y"].agg(
            lambda values: sorted(values.value_counts().loc[lambda counts: counts == counts.max()].index)[0]
        )
        labels.loc[conflicts] = majority.reindex(conflicts)
    keep = ~key.duplicated(keep="first")
    X_out = X.loc[keep].reset_index(drop=True)
    y_out = pd.Series(labels.reindex(key[keep]).to_numpy(), index=X_out.index, name=y.name)
    X_out.attrs["dedupe"] = {
        "rows_in": int(len(X)),
        "rows_out": int(len(X_out)),
        "rows_removed": int(len(X) - len(X_out)),
        "rows_in_duplicate_groups": int(key.duplicated(keep=False).sum()),
        "conflicting_hashes": int(len(conflicts)),
    }
    if meta is None:
        return X_out, y_out
    meta_out = meta.loc[keep].reset_index(drop=True)
    return X_out, y_out, meta_out


def load_deduped(data_dir=None):
    X, y_family, _, meta = load_dataset(data_dir)
    X, y_family, meta = dedupe(X, y_family, meta)
    y_binary = (y_family != BENIGN).astype("int8").rename("binary")
    return X, y_family, y_binary, meta


def group_attack_families(y):
    return y.astype(str).str.replace(r"^DrDoS_", "", regex=True).astype("string")


def split(X, y, seed=42, test_size=0.2):
    return train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed, shuffle=True
    )


def subsample(X, y, cap=CAP, seed=42):
    rng = np.random.default_rng(seed)
    picks = []
    for _, index in y.groupby(y, sort=False).groups.items():
        index = np.asarray(index)
        if len(index) > cap:
            index = rng.choice(index, size=cap, replace=False)
        picks.append(index)
    chosen = np.concatenate(picks)
    rng.shuffle(chosen)
    return X.loc[chosen].reset_index(drop=True), y.loc[chosen].reset_index(drop=True)


def purge_family(X, y, family):
    held = y == family
    keep = ~held
    return (
        X.loc[keep].reset_index(drop=True),
        y.loc[keep].reset_index(drop=True),
        X.loc[held].reset_index(drop=True),
        y.loc[held].reset_index(drop=True),
    )


def drop_constants(X_train, X_test):
    keep = X_train.columns[X_train.nunique() > 1]
    return X_train[keep].copy(), X_test[keep].copy(), list(keep)


def build_pipeline(clf):
    return Pipeline([("scale", StandardScaler()), ("clf", clf)])


def model(name, seed=42, n_estimators=300):
    if name == "dummy":
        return DummyClassifier(strategy="stratified", random_state=seed)
    if name == "logreg":
        return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)
    if name == "random_forest":
        return RandomForestClassifier(
            n_estimators=n_estimators,
            class_weight="balanced",
            n_jobs=-1,
            random_state=seed,
        )
    raise ValueError(name)


def make_models(seed=42, names=None, n_estimators=300):
    names = names or ["dummy", "logreg", "random_forest"]
    return {name: build_pipeline(model(name, seed, n_estimators)) for name in names}


def fpr_at_tpr95(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(np.interp(0.95, tpr, fpr))


def evaluate(y_true, y_pred, y_proba, classes):
    classes = list(classes)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    proba = np.asarray(y_proba)
    if proba.ndim == 1:
        proba = np.column_stack([1.0 - proba, proba])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, average=None, zero_division=0
    )
    rows = []
    for index, label in enumerate(classes):
        binary_true = (y_true == label).astype(int)
        binary_pred = (y_pred == label).astype(int)
        tp = int(((binary_true == 1) & (binary_pred == 1)).sum())
        fp = int(((binary_true == 0) & (binary_pred == 1)).sum())
        tn = int(((binary_true == 0) & (binary_pred == 0)).sum())
        fn = int(((binary_true == 1) & (binary_pred == 0)).sum())
        row = {
            "class": label,
            "precision": float(precision[index]),
            "recall": float(recall[index]),
            "f1": float(f1[index]),
            "support": int(support[index]),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
            "fpr": float(fp / (fp + tn)) if (fp + tn) else np.nan,
            "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "fpr_at_tpr95": np.nan,
        }
        if 0 < binary_true.sum() < len(binary_true):
            row["pr_auc"] = float(average_precision_score(binary_true, proba[:, index]))
            row["roc_auc"] = float(roc_auc_score(binary_true, proba[:, index]))
            row["fpr_at_tpr95"] = fpr_at_tpr95(binary_true, proba[:, index])
        rows.append(row)
    per_class = pd.DataFrame(rows).set_index("class", drop=False)
    per_class.index.name = None
    if len(classes) == 2:
        positive = per_class.iloc[1]
        pr_auc = float(positive["pr_auc"])
        roc_auc = float(positive["roc_auc"])
        fpr95 = float(positive["fpr_at_tpr95"])
        fpr = float(positive["fpr"])
    else:
        pr_auc = float(np.nanmean(per_class["pr_auc"]))
        roc_auc = float(np.nanmean(per_class["roc_auc"]))
        fpr95 = float(np.nanmean(per_class["fpr_at_tpr95"]))
        fpr = float(np.nanmean(per_class["fpr"]))
    return {
        "n_test": int(len(y_true)),
        "n_classes": len(classes),
        "macro_f1": float(f1_score(y_true, y_pred, labels=classes, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=classes, average="weighted", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "fpr": fpr,
        "fpr_at_tpr95": fpr95,
        "per_class": per_class,
    }


def confusion_matrix(y_true, y_pred, labels):
    matrix = sk_confusion_matrix(y_true, y_pred, labels=list(labels))
    return pd.DataFrame(
        matrix,
        index=pd.Index(list(labels), name="true"),
        columns=pd.Index(list(labels), name="pred"),
    )


def fit_and_score(pipeline, X_train, y_train, X_test, y_test):
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    y_pred = pipeline.predict(X_test)
    predict_seconds = time.perf_counter() - start
    y_proba = pipeline.predict_proba(X_test)
    classes = list(pipeline.named_steps["clf"].classes_)
    metrics = evaluate(y_test, y_pred, y_proba, classes)
    return {
        "metrics": metrics,
        "y_pred": np.asarray(y_pred),
        "classes": classes,
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "n_train": int(len(y_train)),
        "n_features_in": int(X_train.shape[1]),
        "n_features_effective": int(X_train.shape[1]),
    }


def metrics_row(metrics, **extra):
    row = {
        "n_test": metrics["n_test"],
        "n_classes": metrics["n_classes"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "accuracy": metrics["accuracy"],
        "pr_auc": metrics["pr_auc"],
        "roc_auc": metrics["roc_auc"],
        "fpr": metrics["fpr"],
        "fpr_at_tpr95": metrics["fpr_at_tpr95"],
    }
    row.update(extra)
    return row


def median_latency_ms(pipeline, X_batch, repeats=5):
    pipeline.predict(X_batch)
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        pipeline.predict(X_batch)
        timings.append((time.perf_counter() - start) * 1000.0)
    return float(np.median(timings))


def run_baseline(seed=42):
    X, y_family, _, _ = load_deduped()
    X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=seed)
    X_train, X_test, features = drop_constants(X_train, X_test)
    y_train = (y_family_train != BENIGN).astype("int8")
    y_test = (y_family_test != BENIGN).astype("int8")
    rows, per_class_rows = [], []
    for name, pipeline in make_models(seed=seed).items():
        result = fit_and_score(pipeline, X_train, y_train, X_test, y_test)
        metrics = result["metrics"]
        rows.append(
            metrics_row(
                metrics,
                model=name,
                model_display=name,
                seed=seed,
                fit_seconds=result["fit_seconds"],
                predict_seconds=result["predict_seconds"],
                n_train=result["n_train"],
                n_features_effective=len(features),
            )
        )
        per_class = metrics["per_class"].copy()
        per_class.insert(0, "model", name)
        per_class.insert(1, "seed", seed)
        per_class_rows.append(per_class)
        confusion_matrix(y_test, result["y_pred"], [0, 1]).to_csv(
            RES / f"baseline_confusion_{name}.csv"
        )
        print(f"{name:14s} macro-F1={metrics['macro_f1']:.4f} FPR={metrics['fpr']:.4f}")
    pd.DataFrame(rows).to_csv(RES / "baseline_binary.csv", index=False)
    pd.concat(per_class_rows, ignore_index=True).to_csv(
        RES / "baseline_binary_per_class.csv", index=False
    )
    return pd.DataFrame(rows)


def run_exp1(seed=42):
    X, y_family, _, _ = load_deduped()
    X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=seed)
    X_train, X_test, _ = drop_constants(X_train, X_test)
    X_train_np, X_test_np, _ = drop_constants(
        X_train.drop(columns=["Protocol"]), X_test.drop(columns=["Protocol"])
    )
    y_binary_train = (y_family_train != BENIGN).astype("int8")
    y_binary_test = (y_family_test != BENIGN).astype("int8")
    y_group_train = group_attack_families(y_family_train)
    y_group_test = group_attack_families(y_family_test)
    classes = sorted(y_group_train.unique())
    tasks = [
        ("binary", X_train, X_test, y_binary_train, y_binary_test, [0, 1]),
        ("multiclass", X_train, X_test, y_group_train, y_group_test, classes),
        ("multiclass_no_protocol", X_train_np, X_test_np, y_group_train, y_group_test, classes),
    ]
    rows, per_class_rows = [], []
    for task, Xtr, Xte, ytr, yte, task_classes in tasks:
        for name, pipeline in make_models(seed=seed).items():
            result = fit_and_score(pipeline, Xtr, ytr, Xte, yte)
            metrics = result["metrics"]
            rows.append(
                metrics_row(
                    metrics,
                    task=task,
                    model=name,
                    model_display=name,
                    seed=seed,
                    fit_seconds=result["fit_seconds"],
                    predict_seconds=result["predict_seconds"],
                    n_train=result["n_train"],
                    n_features_in=result["n_features_in"],
                    n_features_effective=result["n_features_effective"],
                )
            )
            per_class = metrics["per_class"].copy()
            per_class.insert(0, "task", task)
            per_class.insert(1, "model", name)
            per_class.insert(2, "seed", seed)
            per_class_rows.append(per_class)
            confusion_matrix(yte, result["y_pred"], task_classes).to_csv(
                RES / f"exp1_confusion_{task}_{name}.csv"
            )
            print(f"{task:24s} {name:14s} macro-F1={metrics['macro_f1']:.4f}")
    pd.DataFrame(rows).to_csv(RES / "exp1.csv", index=False)
    pd.concat(per_class_rows, ignore_index=True).to_csv(RES / "exp1_per_class.csv", index=False)
    return pd.DataFrame(rows)


def balanced_test(held_X, held_y, X_known_test, y_known_test, seed):
    benign_index = y_known_test.index[y_known_test == BENIGN].to_numpy()
    n = min(len(held_X), len(benign_index))
    rng = np.random.default_rng(seed)
    held_positions = rng.choice(len(held_X), size=n, replace=False) if len(held_X) > n else np.arange(len(held_X))
    benign_positions = rng.choice(benign_index, size=n, replace=False) if len(benign_index) > n else benign_index
    X_attack = held_X.iloc[held_positions].reset_index(drop=True)
    X_benign = X_known_test.loc[benign_positions].reset_index(drop=True)
    X_eval = pd.concat([X_attack, X_benign], ignore_index=True)
    y_eval = np.r_[np.ones(len(X_attack), dtype=np.int8), np.zeros(len(X_benign), dtype=np.int8)]
    return X_eval, y_eval, X_attack


def run_exp2():
    X, y_family, _, meta = load_deduped()
    counts = y_family.value_counts()
    families = sorted(label for label in counts.index if label != BENIGN and counts[label] >= 1_000)
    audit = (
        meta.loc[meta["label"].isin(families)]
        .groupby(["label", "src_file"], observed=True)
        .size()
        .reset_index(name="rows")
        .rename(columns={"label": "family"})
    )
    audit.to_csv(RES / "exp2_purge_audit.csv", index=False)
    rows, per_class_rows, confusion_rows = [], [], []
    diagnostic_seed = SEEDS[0]
    for seed in SEEDS:
        for family in families:
            X_pool, y_pool, held_X, held_y = purge_family(X, y_family, family)
            X_train, X_test, y_train, y_test = split(X_pool, y_pool, seed=seed)
            X_train, X_test, features = drop_constants(X_train, X_test)
            X_train, y_train = subsample(X_train, y_train, cap=CAP, seed=seed)
            y_binary_train = (y_train != BENIGN).astype("int8")
            held_X = held_X[features].copy()
            X_eval, y_eval, X_attack = balanced_test(held_X, held_y, X_test, y_test, seed)
            for name, pipeline in make_models(seed=seed).items():
                result = fit_and_score(pipeline, X_train, y_binary_train, X_eval, y_eval)
                metrics = result["metrics"]
                rows.append(
                    metrics_row(
                        metrics,
                        family=family,
                        seed=seed,
                        model=name,
                        model_display=name,
                        n_family_available=int(len(held_X)),
                        n_test_heldout=int(y_eval.sum()),
                        n_test_benign=int((y_eval == 0).sum()),
                        n_train=result["n_train"],
                        n_features_effective=len(features),
                        fit_seconds=result["fit_seconds"],
                        predict_seconds=result["predict_seconds"],
                        recall_family=float(metrics["per_class"].loc[1, "recall"]),
                    )
                )
                per_class = metrics["per_class"].copy()
                per_class.insert(0, "family", family)
                per_class.insert(1, "model", name)
                per_class.insert(2, "seed", seed)
                per_class_rows.append(per_class)
                confusion_matrix(y_eval, result["y_pred"], [0, 1]).to_csv(
                    RES / f"exp2_confusion_{family}_{name}_s{seed}.csv"
                )
                print(f"{family:16s} {name:14s} seed={seed} macro-F1={metrics['macro_f1']:.3f}")
            if seed == diagnostic_seed:
                known_classes = sorted(y_train.unique().tolist())
                X_diag, y_diag = subsample(X_train, y_train, cap=5_000, seed=seed)
                diag_pipeline = build_pipeline(
                    RandomForestClassifier(
                        n_estimators=100,
                        class_weight="balanced",
                        n_jobs=-1,
                        random_state=seed,
                    )
                )
                diag_pipeline.fit(X_diag, y_diag)
                predicted = diag_pipeline.predict(X_attack)
                table = pd.crosstab(
                    pd.Series([family] * len(predicted)),
                    pd.Series(predicted),
                ).reindex(columns=known_classes, fill_value=0)
                table.index.name = "held_out_family"
                table.columns.name = "predicted_known_family"
                part = table.stack().rename("count").reset_index()
                part.insert(0, "seed", seed)
                confusion_rows.append(part)
    results = pd.DataFrame(rows)
    results.to_csv(RES / "exp2.csv", index=False)
    pd.concat(per_class_rows, ignore_index=True).to_csv(RES / "exp2_per_class.csv", index=False)
    if confusion_rows:
        pd.concat(confusion_rows, ignore_index=True).to_csv(
            RES / "exp2_confusion_to.csv", index=False
        )
    return results


def run_sibling(seed=42):
    X, y_family, _, _ = load_deduped()
    pairs = [
        ("LDAP", ["LDAP", "DrDoS_LDAP"]),
        ("DrDoS_LDAP", ["DrDoS_LDAP", "LDAP"]),
        ("MSSQL", ["MSSQL", "DrDoS_MSSQL"]),
        ("DrDoS_MSSQL", ["DrDoS_MSSQL", "MSSQL"]),
        ("UDP", ["UDP", "DrDoS_UDP"]),
        ("DrDoS_UDP", ["DrDoS_UDP", "UDP"]),
    ]
    rows = []
    for target, purge_labels in pairs:
        held = y_family == target
        keep = ~y_family.isin(purge_labels)
        X_pool = X.loc[keep].reset_index(drop=True)
        y_pool = y_family.loc[keep].reset_index(drop=True)
        held_X = X.loc[held].reset_index(drop=True)
        held_y = y_family.loc[held].reset_index(drop=True)
        X_train, X_test, y_train, y_test = split(X_pool, y_pool, seed=seed)
        X_train, X_test, features = drop_constants(X_train, X_test)
        held_X = held_X[features].copy()
        X_train, y_train = subsample(X_train, y_train, cap=CAP, seed=seed)
        y_binary_train = (y_train != BENIGN).astype("int8")
        X_eval, y_eval, _ = balanced_test(held_X, held_y, X_test, y_test, seed)
        pipeline = make_models(seed=seed, names=["random_forest"])["random_forest"]
        result = fit_and_score(pipeline, X_train, y_binary_train, X_eval, y_eval)
        metrics = result["metrics"]
        rows.append(
            metrics_row(
                metrics,
                target_family=target,
                purged_labels=", ".join(purge_labels),
                seed=seed,
                n_test_heldout=int(y_eval.sum()),
                n_test_benign=int((y_eval == 0).sum()),
                recall_target=float(metrics["per_class"].loc[1, "recall"]),
            )
        )
        print(f"{target:14s} purge={purge_labels!s:35s} macro-F1={metrics['macro_f1']:.4f}")
    out = pd.DataFrame(rows)
    out.to_csv(RES / "exp2_sibling_check.csv", index=False)
    return out


def run_exp3():
    X, y_family, _, _ = load_deduped()
    variants = {"all": None, "75pct": 58, "50pct": 39, "25pct": 20, "top10": 10}
    rows, ranking_tables, selected_rows = [], [], []
    for seed in SEEDS:
        X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=seed)
        X_train, X_test, features = drop_constants(X_train, X_test)
        X_train, y_family_train = subsample(X_train, y_family_train, cap=CAP, seed=seed)
        tasks = {
            "binary": (
                (y_family_train != BENIGN).astype("int8"),
                (y_family_test != BENIGN).astype("int8"),
                [0, 1],
            ),
            "multiclass": (
                group_attack_families(y_family_train),
                group_attack_families(y_family_test),
                sorted(group_attack_families(y_family_train).unique()),
            ),
        }
        for task, (y_train, y_test, classes) in tasks.items():
            mi = pd.Series(
                mutual_info_classif(X_train, y_train, discrete_features=False, random_state=seed),
                index=X_train.columns,
                name="mutual_info",
            )
            forest = RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                n_jobs=-1,
                random_state=seed,
            )
            forest.fit(X_train, y_train)
            importance = pd.Series(forest.feature_importances_, index=X_train.columns, name="rf_importance")
            ranking = pd.concat([mi, importance], axis=1)
            ranking["mi_rank"] = ranking["mutual_info"].rank(ascending=False, method="min").astype(int)
            ranking["rf_rank"] = ranking["rf_importance"].rank(ascending=False, method="min").astype(int)
            ranking.insert(0, "task", task)
            ranking.insert(0, "seed", seed)
            ranking_tables.append(ranking.reset_index().rename(columns={"index": "feature"}))
            for variant, k in variants.items():
                if k is None:
                    pipeline = build_pipeline(
                        RandomForestClassifier(
                            n_estimators=300,
                            class_weight="balanced",
                            n_jobs=-1,
                            random_state=seed,
                        )
                    )
                else:
                    pipeline = Pipeline(
                        [
                            (
                                "select",
                                SelectKBest(
                                    score_func=partial(
                                        mutual_info_classif,
                                        discrete_features=False,
                                        random_state=seed,
                                    ),
                                    k=k,
                                ),
                            ),
                            ("scale", StandardScaler()),
                            (
                                "clf",
                                RandomForestClassifier(
                                    n_estimators=300,
                                    class_weight="balanced",
                                    n_jobs=-1,
                                    random_state=seed,
                                ),
                            ),
                        ]
                    )
                result = fit_and_score(pipeline, X_train, y_train, X_test, y_test)
                latency = median_latency_ms(pipeline, X_test.iloc[:1_000])
                n_effective = k if k is not None else len(features)
                rows.append(
                    metrics_row(
                        result["metrics"],
                        task=task,
                        variant=variant,
                        k_requested=k,
                        model="random_forest",
                        seed=seed,
                        n_train=result["n_train"],
                        n_features_in=result["n_features_in"],
                        n_features_effective=n_effective,
                        fit_seconds=result["fit_seconds"],
                        predict_seconds=result["predict_seconds"],
                        latency_ms_1000=latency,
                        throughput_flows_s=1_000.0 / (latency / 1000.0),
                    )
                )
                confusion_matrix(y_test, result["y_pred"], classes).to_csv(
                    RES / f"exp3_confusion_{task}_{variant}_s{seed}.csv"
                )
                if k is not None:
                    chosen = X_train.columns[pipeline.named_steps["select"].get_support()]
                    selected_rows.extend(
                        {
                            "task": task,
                            "variant": variant,
                            "seed": seed,
                            "feature": feature,
                            "rank": rank,
                        }
                        for rank, feature in enumerate(chosen, start=1)
                    )
                print(f"{task:10s} {variant:6s} k={n_effective:3d} macro-F1={result['metrics']['macro_f1']:.4f}")
    pd.DataFrame(rows).to_csv(RES / "exp3.csv", index=False)
    pd.concat(ranking_tables, ignore_index=True).to_csv(RES / "exp3_feature_ranking.csv", index=False)
    pd.DataFrame(selected_rows).to_csv(RES / "exp3_selected_features.csv", index=False)
    return pd.DataFrame(rows)


In [ ]:
exp1 = run_exp1(seed=42)
results = pd.read_csv(RES / "exp1.csv")
per_class = pd.read_csv(RES / "exp1_per_class.csv")
display(results[["task", "model", "n_classes", "macro_f1", "weighted_f1", "accuracy", "pr_auc", "roc_auc", "fpr", "fpr_at_tpr95", "fit_seconds", "n_features_effective"]].round(4))

In [ ]:
order = ["binary", "multiclass", "multiclass_no_protocol"]
labels = {
    "binary": "Benign / DDoS",
    "multiclass": "13 classes",
    "multiclass_no_protocol": "13 classes, no Protocol",
}
plot = results.copy()
plot["task_label"] = plot["task"].map(labels)
plot["task_label"] = pd.Categorical(plot["task_label"], categories=[labels[t] for t in order], ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, metric in zip(axes, ["macro_f1", "weighted_f1"]):
    sns.barplot(data=plot, x="task_label", y=metric, hue="model", ax=ax)
    ax.set_ylim(0, 1.02)
    ax.set_title(metric.replace("_", "-").title())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
    ax.legend(title="", fontsize=8)
fig.suptitle("H1 : mêmes lignes, mêmes modèles, seule la cible change", y=1.03)
fig.tight_layout()
fig.savefig(FIG / "03_exp1_macro_weighted_f1.png", bbox_inches="tight")
plt.show()

display(
    results.pivot(index="task", columns="model", values="macro_f1")
    .reindex(order)
    .round(4)
)

In [ ]:
# Rappel par classe pour le modèle multiclasses le plus fort.
focus = per_class[
    (per_class["task"] == "multiclass") & (per_class["model"] == "random_forest")
].copy()
focus["class"] = focus["class"].astype(str)
focus = focus.sort_values("recall")

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=focus, x="recall", y="class", color="#4c72b0", ax=ax)
ax.set_xlim(0, 1.02)
ax.set_title("Rappel par classe — forêt aléatoire multiclasses")
ax.set_xlabel("rappel")
ax.set_ylabel("")
for i, row in enumerate(focus.itertuples()):
    ax.text(row.recall + 0.01, i, f"{row.recall:.3f}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "03_exp1_per_class_recall.png", bbox_inches="tight")
plt.show()

display(focus[["class", "precision", "recall", "f1", "support", "fpr"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
titles = ["binary", "multiclass", "multiclass_no_protocol"]
for ax, task in zip(axes, titles):
    matrix = pd.read_csv(
        RES / f"exp1_confusion_{task}_random_forest.csv", index_col=0
    )
    sns.heatmap(matrix, cmap="Blues", cbar=False, ax=ax)
    ax.set_title(task)
    ax.set_xlabel("prédit")
    ax.set_ylabel("réel")
fig.suptitle("Matrices de confusion — binaire (2×2) et multiclasses (13×13), forêt aléatoire", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "03_exp1_confusion.png", bbox_inches="tight")
plt.show()

In [ ]:
# Ablation de Protocol : retirer Protocol dégrade-t-il le score multiclasses ?
base = results[results["task"] == "multiclass"].set_index("model")
abl = results[results["task"] == "multiclass_no_protocol"].set_index("model")
delta = (abl["macro_f1"] - base["macro_f1"]).rename("delta_macro_f1_no_protocol")
display(delta.round(4).to_frame())

base_pc = per_class[
    (per_class["task"] == "multiclass") & (per_class["model"] == "random_forest")
].set_index("class")
abl_pc = per_class[
    (per_class["task"] == "multiclass_no_protocol")
    & (per_class["model"] == "random_forest")
].set_index("class")
delta_recall = (abl_pc["recall"] - base_pc["recall"]).sort_values()
display(delta_recall.round(4).to_frame())

fig, ax = plt.subplots(figsize=(10, 5))
delta_recall.plot(kind="barh", ax=ax, color=np.where(delta_recall < 0, "#c44e52", "#55a868"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Δ rappel (sans Protocol − avec Protocol)")
ax.set_title("Ablation de Protocol par classe — forêt aléatoire")
fig.tight_layout()
fig.savefig(FIG / "03_exp1_protocol_ablation.png", bbox_inches="tight")
plt.show()

## Bilan de H1

**Observation.** Le Macro-F1 passe de presque 1,0 en binaire à environ 0,60–0,76 pour les deux modèles multiclasses entraînés, tandis que le Weighted-F1 reste au-dessus de 0,94.  L’écart entre Macro-F1 et Weighted-F1 traduit le déséquilibre : les familles rares sont manquées alors que les grandes familles maintiennent le score pondéré.  Retirer `Protocol` change le Macro-F1 de moins de 0,01.

**Interprétation ML.** H1 est soutenue.  Un modèle peut être quasi parfait pour « attaque / pas attaque » et beaucoup plus faible pour « quelle attaque ».  Le faible écart lié à Protocol est rassurant : le modèle multiclasses ne lit pas seulement le protocole de transport, car les caractéristiques de taille de paquet et de temporalité portent l’essentiel du signal.  Le rappel par classe est la vue honnête ; la matrice de confusion 13×13 montre quelles familles sont fusionnées.

**Interprétation cyber.** Réponse à Q2 : non, un détecteur binaire n’est pas nécessairement utile pour identifier une attaque.  Un analyste a besoin du étiquette de famille pour choisir une mitigation (SYN flood vs réflexion DNS vs amplification TFTP).  Une alerte binaire est une porte de triage, pas un produit de classification.  L’ablation de Protocol suggère que le classifieur apprend la forme du flux, mais la confusion restante entre familles proches constitue la limite opérationnelle.

**Limite.** La correspondance à 13 classes fusionne `DrDoS_X` avec `X` ; cette expérience ne teste donc pas si les variantes de réflexion et d’exploitation sont séparables.  C’est le rôle plus fin de l’expérience 2.  Les données de laboratoire et une seule graine limitent la validité externe.
